## models

In [92]:
import quante as qt

# xxx
qt.generate.matrix.heisenberg_matrix(
    L=10, j=(1.,1.,1.), hz=0., pauli=True
)

array([[9., 0., 0., ..., 0., 0., 0.],
       [0., 7., 2., ..., 0., 0., 0.],
       [0., 2., 5., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 5., 2., 0.],
       [0., 0., 0., ..., 2., 7., 0.],
       [0., 0., 0., ..., 0., 0., 9.]])

In [93]:
# Dirac Fermion Model

import quante as qt

qt.generate.matrix.models.syk.syk4_dirac(
    L=10, Nf=5, J=1.
)

array([[-0.00096016+0.j        ,  0.05940527+0.05938672j, -0.00391743+0.04284224j, ...,  0.        +0.j        ,  0.        +0.j        ,  0.        +0.j        ],
       [ 0.05940527-0.05938672j, -0.04984348+0.j        ,  0.0085327 -0.04090801j, ...,  0.        +0.j        ,  0.        +0.j        ,  0.        +0.j        ],
       [-0.00391743-0.04284224j,  0.0085327 +0.04090801j,  0.03182612+0.j        , ...,  0.        +0.j        ,  0.        +0.j        ,  0.        +0.j        ],
       ...,
       [ 0.        +0.j        ,  0.        +0.j        ,  0.        +0.j        , ...,  0.05455661+0.j        , -0.00467614-0.01433816j, -0.05723523-0.02644771j],
       [ 0.        +0.j        ,  0.        +0.j        ,  0.        +0.j        , ..., -0.00467614+0.01433816j,  0.24523838+0.j        ,  0.00976024+0.06160549j],
       [ 0.        +0.j        ,  0.        +0.j        ,  0.        +0.j        , ..., -0.05723523+0.02644771j,  0.00976024-0.06160549j,  0.24156601+0.j        ]])

## 基本使用

定义哈密顿量：

$$
    H = \sum_{i = 1}^{L - 1} (\sigma^{x}_{i}\sigma^{x}_{i + 1} + \sigma^{y}_{i} \sigma^{y}_{i + 1} + \frac{1}{2}  \sigma^{z}_{i} \sigma^{z}_{i + 1})
$$

In [ ]:
import quante as qt
op = qt.generate.operas.spin
L = 4
builder = op.builder()
for i in range(L-1):
    builder += 'xx', [i, i+1], 1.
    builder += 'yy', [i, i+1], 1.
    builder += 'zz', [i, i+1], 1.
ham = builder.build()

basis = qt.generate.basis.spin_basis(L=L, Nup=L//2)
hammat = ham.to_matrix(basis=basis, pauli=True, check_symm=True)
hammat

U(1) check passed


{}

生成基矢

In [3]:
# 生成具有粒子数生活和动量守恒的基矢
basis = qt.generate.basis.spin_basis(L=4, Nup=2, kblock=0)

# 可以查看基矢的个数：
print("空间维数", basis.Ns)

# # 可以获得某个基矢在全空间中的表示
state = basis.to_full_space(0)

# 可以可视化全空间的基矢：
print("第 0 个基矢：")
basisfull = qt.generate.basis.spin_basis(L=4)
basisfull.show_state(basis[0])

空间维数 2
第 0 个基矢：
↑↑↓↓: (0.5+0j)
↑↓↓↑: (0.5+0j)
↓↑↑↓: (0.5+0j)
↓↓↑↑: (0.5+0j)


In [4]:
# 获得哈密顿量在给定基矢下的矩阵：
mat = ham.to_matrix(basis, pauli=True)
mat

array([[ 1.        +0.j,  4.24264069+0.j],
       [ 2.82842712+0.j, -3.        +0.j]])

计算基态能

In [5]:
# 对角化
engs, eigstates = qt.linalg.eigh(mat, k=1)  # 获得最低能量的本征态
engs

array([-4.46410162])

计算纠缠

In [6]:
# 计算纠缠：
entspect = qt.measure.entanglement_spectrum(eigstates[:,0], L//2, basis)  # 纠缠谱
entspect
qt.measure.entropy(entspect)

np.float64(1.1741418371072039)

# Lindbladian

In [1]:
import numpy as np
import quante.generate.operas as op

L, gm = 4, 0.5
ham = op.heisenberg_operator(L=L, j=1.)
jump_ops = [np.sqrt(gm) * op.m(i) for i in range(L)]
liou = op.Lindbladian(L, ham, jump_ops)
liou.show()

Lindbladian Operator in vectorized space, L=4, indx_order=stacked, flip=False
Lindbladian at 0x1670b6fb350, 
|   x     x       coef. |   y     y       coef. |   z     z       coef. |   p     m       coef. |
|-----------------------|-----------------------|-----------------------|-----------------------|
|   0     1     -1.000j |   0     1     -1.000j |   0     1     -1.000j |   0     0      -0.250 |
|   1     2     -1.000j |   1     2     -1.000j |   1     2     -1.000j |   1     1      -0.250 |
|   2     3     -1.000j |   2     3     -1.000j |   2     3     -1.000j |   2     2      -0.250 |
|   4     5      1.000j |   4     5      1.000j |   4     5      1.000j |   3     3      -0.250 |
|   5     6      1.000j |   5     6      1.000j |   5     6      1.000j |   4     4      -0.250 |
|   6     7      1.000j |   6     7      1.000j |   6     7      1.000j |   5     5      -0.250 |
|                                                                       |   6     6      -0.250 |
|        

In [2]:
import quante as qt

# using trivial basis
basis = qt.generate.basis.spin_basis(2*L)
liou_mat = liou.to_matrix(basis, pauli=True, check_symm=True)
liou_mat

array([[-2.  +0.j,  0.  +0.j,  0.  +0.j, ...,  0.  +0.j,  0.  +0.j,  0.  +0.j],
       [ 0.  +0.j, -1.75-2.j,  0.  +2.j, ...,  0.  +0.j,  0.  +0.j,  0.  +0.j],
       [ 0.  +0.j,  0.  +2.j, -1.75-4.j, ...,  0.  +0.j,  0.  +0.j,  0.  +0.j],
       ...,
       [ 0.  +0.j,  0.  +0.j,  0.  +0.j, ..., -0.25-4.j,  0.  +2.j,  0.  +0.j],
       [ 0.  +0.j,  0.  +0.j,  0.  +0.j, ...,  0.  +2.j, -0.25-2.j,  0.  +0.j],
       [ 0.  +0.j,  0.  +0.j,  0.  +0.j, ...,  0.  +0.j,  0.  +0.j,  0.  +0.j]])

In [38]:
# spin_basis_2d allows various symmetries (for now, only Z2 symmetries are supported)
basis = qt.generate.basis.spin_basis_2d(Lx=L, Ly=2, Ndiff=(range(L,2*L), 0), pxblock=0)
liou_mat = liou.to_matrix(basis, pauli=True, check_symm=True)
print(liou_mat.dtype)
eng1 = qt.linalg.sortcomplex(np.linalg.eigvals(liou_mat))

pxblock check passed


complex128


In [39]:
# spin_basis_2d allows various symmetries (for now, only one Z2 symmetries are supported)
basis = qt.generate.basis.spin_super_basis(L, Ndiff=0, pblock=0)
liou_mat = liou.to_matrix(basis, pauli=True, check_symm=True)
eng2 = qt.linalg.sortcomplex(np.linalg.eigvals(liou_mat))
np.allclose(eng2, eng1)

pblock check passed


True

for now, the spin_basis_2d only supports Z2 symmetry, and spin_super_basis only support *one* Z2 symmetry.

for more complicated symmetries, use quspin

In [42]:
import quante.bridge.quspin_utils as qs

L, gm = 4, 0.5
ham = op.heisenberg_operator(L=L, j=1.)
jump_ops = [np.sqrt(gm) * op.m(i) for i in range(L)]
liou = op.super_oper.Lindbladian(L, ham, jump_ops)

basis = qs.spin_super_basis(L, pblock=0, pauli=True)
liou_mat = qs.hamiltonian(liou, basis, dtype=np.complex128).toarray()
print(liou_mat.dtype) # should be complex128

# we can also realify it by basis (using projection matrix)
liou_real_mat = basis.realify(liou_mat)
print(liou_real_mat.dtype) # should be float64

# benchmark
eng1 = qt.linalg.sortcomplex(np.linalg.eigvals(liou_mat))
eng2 = qt.linalg.sortcomplex(np.linalg.eigvals(liou_real_mat))
np.allclose(eng2, eng1)

complex128
float64


True

In [43]:
# benchmark with quante
basis = qt.generate.basis.spin_super_basis(L, pblock=0)
liou_mat = liou.to_matrix(basis, pauli=True, check_symm=True)
eng3 = qt.linalg.sortcomplex(np.linalg.eigvals(liou_mat))
np.allclose(eng2, eng3)

pblock check passed


True

other function of liouville operator

In [8]:
# get non-Hamiltonian part of Lindbladian
nonham = liou.nonhermitian_part()
nonham.show()

SpinOper at 0x14ee8d271a0, 
|   x     x       coef. |   y     y       coef. |   z     z       coef. |   p     m       coef. |
|-----------------------|-----------------------|-----------------------|-----------------------|
|   0     1       1.000 |   0     1       1.000 |   0     1       1.000 |   0     0     -0.250j |
|   1     2       1.000 |   1     2       1.000 |   1     2       1.000 |   1     1     -0.250j |
|   2     3       1.000 |   2     3       1.000 |   2     3       1.000 |   2     2     -0.250j |
|                                                                       |   3     3     -0.250j |



## 梯子形系统

```
      0   2   4   6
   ---◻---◻---◻---◻---
      |   |   |   |
   ---◻---◻---◻---◻---
      1   3   5   7
```

In [99]:
j1, j2, j3 = 1.0, 2.0, 1.0
import quante as qt
op = qt.generate.operas.spin

L = 5

H_Spart = j1 * op.sum(op.xx(2*i,2*i+2) + op.yy(2*i,2*i+2) + op.zz(2*i,2*i+2) for i in range(L-1))
H_Lpart = op.sum(j2 * (op.xx(2*i+1,2*i+3) + op.yy(2*i+1,2*i+3)) + j1 * op.zz(2*i+1,2*i+3) for i in range(L-1))
H_SLpart = j3 * op.sum(op.xx(2*i,2*i+1) + op.yy(2*i,2*i+1) + op.zz(2*i,2*i+1) for i in range(L))

H = H_Spart + H_Lpart + H_SLpart
H

x 方向和 y 方向都是 zzx 相互作用：

In [100]:
j1, j2, j3 = 1.0, 2.0, 1.0

for L in range(4, 10):
    
    H_Spart = j1 * op.sum(op.xx(2*i,2*i+2) + op.yy(2*i,2*i+2) + op.zz(2*i,2*i+2) for i in range(L-1))
    H_Lpart = op.sum(j2 * (op.xx(2*i+1,2*i+3) + op.yy(2*i+1,2*i+3)) + j1 * op.zz(2*i+1,2*i+3) for i in range(L-1))
    H_SLpart = j3 * op.sum(op.xx(2*i,2*i+1) + op.yy(2*i,2*i+1) + op.zz(2*i,2*i+1) for i in range(L))
    
    H = H_Spart + H_Lpart + H_SLpart
    
    basis = qt.generate.basis.spin_basis(L=2*L, Nup=L)
    mat = H.to_matrix(basis, pauli=False, sparse=True)
    gdeng = qt.linalg.eigvalsh(mat, k=1)[0]
    print(f"L={L}, ground state energy={gdeng}")

L=4, ground state energy=-5.149175306097196
L=5, ground state energy=-6.5495393167336005
L=6, ground state energy=-7.96778141501053
L=7, ground state energy=-9.379011867020704
L=8, ground state energy=-10.793329596691677
L=9, ground state energy=-12.206480865439978


与 quspin 的转换

In [101]:
# 对比 quspin 和 quante 的效率（需要在安装 quspin 的环境中运行）
import quante as qt
import quante.bridge.quspin_utils as qs
import numpy as np

L = 20
ham = qt.generate.operas.spin.heisenberg_operator(L, j=(1, 1, 1))
ham = ham.expandxy(pauli=False)
quspin_basis = qs.spin_basis(L=L, pauli=0)

with qt.basicfun.Timer("quspin time: "):
    mat1 = qs.hamiltonian(ham, quspin_basis, dtype=np.float64)
    # faster on linux with omp

basis = qt.generate.basis.spin_basis(L=L)
with qt.basicfun.Timer("quante time: "):
    mat2 = ham.to_matrix(basis, pauli=False, sparse=True)

print("diff: ",qt.linalg.norm(mat1 - mat2))

quspin time: : 2.1298786999977892 seconds
quante time: : 0.42843070000526495 seconds


diff:  0.0


生成矩阵的难点在于, 稀疏矩阵的加法, 它无法利用并行加速

automata 之所以更快是因为, 它最小化了大型稀疏矩阵加法的次数

eigvalsh (real)
|  dim\backend   |  numpy (syevd)  |  scipy (syevd) |  scipy (syevr) |  torch  |  torch-cuda  |  matlab  |  matlab-gpu  |  julia (syevd)   |  julia (syevd-cuda)   |  matrix ocupied  |  memory needed  |
|:--------------:|:---------------:|:--------------:|:--------------:|:-------:|:------------:|:--------:|:------------:|:----------------:|:---------------------:|:----------------:|:---------------:|
|  2^13 = 8192   |      8s         |      10s       |      10s       |   7s    |      <font color="red">3s</font>      |    19s   |     16s      |       8s         |           ?           |       0.5 G      |      1 G        |
|  2^14 = 16384  |      34s        |      41s       |      41s       |   37s   |      <font color="red">17s</font>|    43s   |     39s      |       34s        |          17s          |       2 G        |      4 G        |
|  2^15 = 32768  |     242s        |      292s      |      282s      |   266s  |       x      |   288s   |    <font color="red">184s</font>      |      242s        |           x           |       8 G        |     16 G        |
|  2^16 = 65536  |     1795s       |      1894s     |      1964s     |   1950s |       x      |   1827s  |      ?       |      <font color="red">1758s</font>       |           x           |      32 G        |     64 G        |



eigvalsh (complex)
|  dim\backend   |  numpy (zhevd)  |  scipy (zhevd) |  scipy (zhevr) |  torch  |  torch-cuda  |  matlab  |  matlab-gpu  |  julia (syevd)   |  julia (syevd-cuda)   |  matrix ocupied  |  memory needed  |
|:--------------:|:---------------:|:--------------:|:--------------:|:-------:|:------------:|:--------:|:------------:|:----------------:|:---------------------:|:----------------:|:---------------:|
|  2^13 = 8192   |      14s        |      15s       |      15s       |   14s   |      <font color="red">8s</font>      |    20s   |     26s      |       13s        |           ?           |       1 G        |      2 G        |
|  2^14 = 16384  |      95s        |      103s      |      104s      |   97s   |      <font color="red">54s</font>     |   113s   |    101s      |      93s         |         55s           |       4 G        |      8 G        |
|  2^15 = 32768  |      <font color="red">675s</font>       |      778s      |      716s      |   732s  |       ?      |   775s   |    ?         |      696s        |          x            |       16 G       |     32 G        |
|  2^16 = 65536  |     ?           |      ?         |      ?         |   ?     |       x      |   ?      |      ?       |      ?           |           x           |       64 G       |    128 G        |

eigh (real)
|  dim\backend   |  numpy (syevd)  |  scipy (syevd) |  scipy (syevr) |  torch  |  torch-cuda  |  matlab  |  matlab-gpu  |  julia (syevd) |  julia (syevd-cuda)   |
|:--------------:|:---------------:|:--------------:|:--------------:|:-------:|:------------:|:--------:|:------------:|:--------------:|:---------------------:|
|  2^13 = 8192   |        22s      |        22s     |     51s        |    22s  |     <font color="red">5s</font>       |   29s    |      x       |     21s        |          ?            |
|  2^14 = 16384  |       130s      |      148s      |     1194s      |   158s  |      <font color="red">34s</font>     |  166s    |      x       |     139s       |          33s          |
|  2^15 = 32768  |        x        |       x        |     24600s     |    x    |       x      |  1440s   |      x       |     <font color="red">1200s</font>      |           x           |
|  2^16 = 65536  |        ?        |       ?        |       ?        |    ?    |       x      |    ?     |      x       |       ?        |           x           |

eigh (complex)
|  dim\backend   |  numpy (zhevd)  |  scipy (zhevd) |  scipy (zhevr) |  torch  |  torch-cuda  |  matlab  |  matlab-gpu  |  julia (syevd) |  julia (syevd-cuda)   |
|:--------------:|:---------------:|:--------------:|:--------------:|:-------:|:------------:|:--------:|:------------:|:--------------:|:---------------------:|
|  2^13 = 8192   |       97s       |      101s      |     107s       |   100s  |      <font color="red">18s</font>     |   108s   |      x       |     98s        |
|  2^14 = 16384  |       710s      |      747s      |     771s       |   760s  |      <font color="red">126s</font>    |  764s    |      x       |     737s       |          128s         |
|  2^15 = 32768  |        x        |       x        |     6050s      |    x    |       x      |  5978s   |      x       |     <font color="red">5856s</font>      |           x           |
|  2^16 = 65536  |        ?        |       ?        |       ?        |    ?    |       x      |    ?     |      x       |       ?        |           x           |

svdvals (real)
|  dim\backend   |  numpy (gesdd)  |  scipy (gesdd) |  scipy (gesvd) |  torch  |  torch (gesvd-cuda)  |  matlab  |  matlab-gpu  |  julia (gesdd)   |  julia (gesvd-cuda)   |
|:--------------:|:---------------:|:--------------:|:--------------:|:-------:|:--------------------:|:--------:|:------------:|:----------------:|:---------------------:|
|  2^13 = 8192   |        13s      |    17s         |      17s       |   13s   |       <font color="red">8s</font>             |   132s   |   21s        |      16s         |      ?                |
|  2^14 = 16384  |       79s       |    114s        |     114s       | 102s    |     <font color="red">47s</font>              |  965s    |  72s         | 107s             |      ?                |
|  2^15 = 32768  |       538s      |    757s        |     743s       | 721s    |    x                 |  7839s   |   <font color="red">435s</font>       |  750s            |      ?                |
|  2^16 = 65536  |        ?        |       ?        |       ?        |    ?    |       x              |    ?     |      ?       |       ?          |           ?           |

svdvals (complex)
|  dim\backend   |  numpy (gesdd)  |  scipy (gesdd) |  scipy (gesvd) |  torch  |  torch (gesvd-cuda)  |  matlab  |  matlab-gpu  |  julia (gesdd)   |  julia (gesvd-cuda)   |  matrix ocupied  |  memory needed  |
|:--------------:|:---------------:|:--------------:|:--------------:|:-------:|:--------------------:|:--------:|:------------:|:----------------:|:---------------------:|:----------------:|:---------------:|
|  2^13 = 8192   |      29s        |      34s       |      34s       |   32s   |       <font color="red">17s</font>            |   33s    |      26s     |      28s         |           18s         |       1 G        |    4 G          |
|  2^14 = 16384  |      204s       |      223s      |      229s      |   229s  |      <font color="red">117s</font>           |   247s   |    168s      |      225s        |         118s          |       4 G        |      16 G       |
|  2^15 = 32768  |      <font color="red">1685s</font>      |      1866s     |      1837s     |   1858s |       ?              |   1800s  |    ?         |      1797s       |         ?             |       16 G       |     64 G        |
|  2^16 = 65536  |        ?        |       ?        |       ?        |    ?    |       ?              |    ?     |      ?       |       ?          |           ?           |       64 G       |    256 G        |


eigvals (real)
|  dim\backend   |  numpy          |  scipy         |   torch    |  torch-cuda  |  matlab    |  matlab-gpu  |   julia  |
|:--------------:|:---------------:|:--------------:|:----------:|:------------:|:----------:|:------------:|:--------:|
|  2^13 = 8192   |      117s       |   140s         |  133s      |  <font color="red">70s</font>         |   133s     |   69s        |  120s    |
|  2^14 = 16384  |    698s         |   1164s        |  883s      |  467s        |   1014s    |   <font color="red">340s</font>|  748s    |
|  2^15 = 32768  |    4923s        |   6176s        |  5390s     |  <font color="red">1690s</font>       |   5459s    |   1703s      |  5031s   |
|  2^16 = 65536  |        ?        |       ?        |    ?       |    ?         |       ?    |    ?         |      ?   |

eigvals (complex)
|  dim\backend   |  numpy          |  scipy         |   torch    |  torch-cuda  |  matlab    |  matlab-gpu  |   julia  |
|:--------------:|:---------------:|:--------------:|:----------:|:------------:|:----------:|:------------:|:--------:|
|  2^13 = 8192   |      210s       |     253s       |    264s    |  <font color="red">149s</font>       |    255s    |   140s       |  224s    |
|  2^14 = 16384  |      1431s      |     1672s      |    1670s   |  758s        |    1612s   |   <font color="red">710s</font>       |  1461s   |
|  2^15 = 32768  |    10563s       |     11751s     |    11191s  |  ?           |   11188s   |    ?         | <font color="red">10446s</font>   |
|  2^16 = 65536  |        ?        |       ?        |    ?       |    ?         |       ?    |    ?         |      ?   |

eig (real)
|  dim\backend   |  numpy          |  scipy         |   torch    |  torch-cuda  |  matlab    |  matlab-gpu  |   julia  |
|:--------------:|:---------------:|:--------------:|:----------:|:------------:|:----------:|:------------:|:--------:|
|  2^13 = 8192   |       <font color="red">167s</font>      |    183s        |   190s     |  178         |   189s     |   x          |  188s    |
|  2^14 = 16384  |       <font color="red">1166s</font>     |    1259s       |   1292s    | ?(1404s)     |   1279s    |   x          |  1310s   |
|  2^15 = 32768  |       <font color="red">9871s</font>     |       9774s    |    8770s   |    ?         |    9121s   |    x         |   9871s  |
|  2^16 = 65536  |        ?        |       ?        |    ?       |    ?         |       ?    |    x         |      ?   |

eig (complex)
|  dim\backend   |  numpy          |  scipy         |   torch    |  torch-cuda  |  matlab    |  matlab-gpu  |   julia  |
|:--------------:|:---------------:|:--------------:|:----------:|:------------:|:----------:|:------------:|:--------:|
|  2^13 = 8192   |      <font color="red">286s</font>       |    370s        |    373s    |  607s        |  466s      |   x          |  485s    |
|  2^14 = 16384  |      <font color="red">2241s</font>      |    2579s       |    2341s   |  4617s       |  3307s     |   x          |  3340s   |
|  2^15 = 32768  |        16402s   |       ?        |    ?       |    ?         |       ?    |    x         |      ?   |
|  2^16 = 65536  |        ?        |       ?        |    ?       |    ?         |       ?    |    x         |      ?   |


上述测试的设备信息：
```text
CPU 信息：
Intel64 Family 6 Model 165 Stepping 5, GenuineIntel
CPU 核心数： 20
GPU 信息：
NVIDIA GeForce RTX 3090
GPU 数量： 1
Wed Sep  3 11:31:55 2025
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.94                 Driver Version: 560.94         CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3090      WDDM  |   00000000:01:00.0  On |                  N/A |
|  0%   45C    P8             26W /  350W |    1940MiB /  24576MiB |     12%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+------------------------+----------------------+

numpy 版本： 2.1.3
torch 版本： 2.6.0+cu124
```


The failure of numpy is due to the dimension of workspace required by lapack-syevd overflowing int32

We can use scipy syevr instead, but much slower.

In [102]:
# import quante as qt
# import torch as tc
# import scipy as sp
# import numpy as np

# L = 10
# ham = qt.generate.operas.spin.heisenberg_operator(L, j=(1, 1, 1))
# basis = qt.generate.basis.spin_basis(L=L)
# mat = ham.to_matrix(basis, pauli=False, sparse=True)

# with qt.basicfun.Timer("numpy"):
#     res = np.linalg.eigvalsh(mat.toarray())
#     print(res)

# from scipy.linalg.lapack import dsyevd
# with qt.basicfun.Timer("scipy(evd)"):
#     res = sp.linalg.eigvalsh(mat.toarray(), driver='evd')
#     print(res)

# with qt.basicfun.Timer("scipy(evr)"):
#     res = sp.linalg.eigvalsh(mat.toarray(), driver='evr')
#     print(res)

# with qt.basicfun.Timer("torch(cpu)"):
#     res = tc.linalg.eigvalsh(tc.tensor(mat.toarray()))
#     print(res)

# with qt.basicfun.Timer("torch(cuda)"):
#     res = tc.linalg.eigvalsh(tc.tensor(mat.toarray(), device='cuda'))
#     print(res)

# with qt.basicfun.Timer("matlab(cpu)"):
#     res = qt.linalg.eigvalsh(mat, backend='matlab', usegpu=False)
#     print(res)

# with qt.basicfun.Timer("matlab(cuda)"):
#     res = qt.linalg.eigvalsh(mat, backend='matlab', usegpu=True)
#     print(res)

# # from julia import Main
# # Main.eval("using LinearAlgebra")
# # Main.eval("using MKL")
# # Main.mat = mat.toarray()
# # with qt.basicfun.Timer("julia"):
# #     res = Main.eval("LAPACK.syevd!('N', 'U', mat)")  
# #     # 'N' means only compute eigenvalues, 'V' means compute eigenvectors
# #     print(res)


In [103]:
# import platform
# import os

# print("CPU 信息：")
# print(platform.processor())
# print("CPU 核心数：", os.cpu_count())

# import torch
# print("GPU 信息：")
# print(torch.cuda.get_device_name(0))
# print("GPU 数量：", torch.cuda.device_count())

# import subprocess
# output = subprocess.check_output("nvidia-smi", shell=True, encoding="utf-8")
# print(output)

# import numpy as np
# print("numpy 版本：", np.__version__)
# print("torch 版本：", torch.__version__)


## SU(2) 工具

下面函数中参数中 `jmblock = (J, m)`

`J` 可取的值，可取 `L/2`, `L2/2-1`, ... `0`，`m` 可取的值为 `5`, `4`, `3`, `2`, `1`, `0`, `-1`, `-2`, `-3`, `-4`, `-5`

`dim` 表示子空间的维数，`num` 表示子空间重复的次数，因而：

In [104]:
import quante as qt
L = 4
basis = qt.generate.basis.spin_basis(L, jmblock=(2, 2))

basis.print_dims(L)

   J  |   num  |   dim   
-----------------------
  2.0 |   5    |  1
  1.0 |   3    |  3
  0.0 |   1    |  2
-----------------------
note: \sum num * dim = 2^L


可以与普通的 basis 一样生成矩阵，但目前采用投影矩阵的方法，效率低

In [105]:
ham = qt.generate.operas.spin.heisenberg_operator(L)
mat = ham.to_matrix(basis, pauli=False)
mat.shape, mat

((1, 1), array([[0.75]]))

In [106]:
basis_ = qt.generate.basis.spin_basis(L)
mat_ = ham.to_matrix(basis_, pauli=False)
qt.linalg.eigvalsh(mat_).reshape(4,-1)

array([[-1.6160254 , -0.95710678, -0.95710678, -0.95710678],
       [-0.25      , -0.25      , -0.25      ,  0.1160254 ],
       [ 0.45710678,  0.45710678,  0.45710678,  0.75      ],
       [ 0.75      ,  0.75      ,  0.75      ,  0.75      ]])

可以看到 0.75 确实重复的 5 次

In [107]:
basis = qt.generate.basis.spin_basis(L, jmblock=(1, 1))
mat = ham.to_matrix(basis, pauli=False)
qt.linalg.eigvalsh(mat)

array([-0.95710678, -0.25      ,  0.45710678])

对比可以看到 这三个数每个都重复了三次

验证每个基矢都是 $J^2$ 的本征态

In [108]:
vec = basis.to_full_space(1)  # 第二个基矢，任何一个基矢都满足
# 这个向量是 J^2 的本征态

op = qt.generate.operas
op_Jx = op.sum(op.x(i) for i in range(L))
op_Jy = op.sum(op.y(i) for i in range(L))
op_Jz = op.sum(op.z(i) for i in range(L))
op_J2 = op_Jx**2 + op_Jy**2 + op_Jz**2

basis_ = qt.generate.basis.spin_basis(L)
mat_J2 = op_J2.to_matrix(basis_, pauli=False)

import numpy as np
np.real_if_close(mat_J2 @ vec - 1*(1+1)*vec)  # 这个向量是 J^2 的本征态

array([ 0.,  0., -0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.])

验证每个基矢都是 $J_z$ 的本征态

In [109]:
basis_ = qt.generate.basis.spin_basis(L)
mat_Jz = op_Jz.to_matrix(basis_, pauli=False)

import numpy as np
np.real_if_close(mat_Jz @ vec - 1*vec)  # 这个向量是 Jz 的本征态

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])

## 算符代数的一些工具

目前主要是费米子的算符代数，后续会添加玻色子算符代数的内容。

In [110]:
import quante as qt
op = qt.generate.operas.fermion

L = 4
builder = op.builder()
for l in range(L):
    builder += '+-', [l, l+1], 1.
    builder += '+-', [l+1, l], 1.
ham = builder.build()
ham

In [111]:
import quante as qt
op = qt.generate.operas.fermion

builder = op.builder()
builder += '-'*6 + '+'*6, list(range(1, 7))+list(range(1, 7)), 1.
ham = builder.build()
ham


In [112]:
ham.normal_ordering()

## JW transformation

In [113]:
# JW transformation  spin -> fermion
import quante as qt

op = qt.generate.operas.fermion
builder = op.builder()
L = 10
J, γ = 1, 0.0
for i in range(L-1):
    builder += "+-", [i+1, i], (J+γ)/2
    builder += "-+", [i+1, i], -(J-γ)/2
ham = builder.build()
ham.normal_ordering()
# ham.jw_transfer()

In [114]:
# JW transformation  fermion -> spin

import quante as qt

op = qt.generate.operas.fermion
builder = op.builder()
L = 10
J, γ = 1, 0.0
for i in range(L-1):
    builder += "+-", [i+1, i], (J+γ)/2
    builder += "+-", [i, i+1], (J-γ)/2
ham = builder.build()
ham

In [1]:
# 验证正确性：
import quante as qt
import numpy as np

op = qt.generate.operas.spin

L = 5
basis = qt.generate.basis.spin_basis(L=L)

ham = op.heisenberg_operator(L=L).expandxy(pauli=True)

mat1 = ham.to_matrix(basis, pauli=True)

ham = ham.jw_transfer(pauli=True)  # spin -> fermion
ham = ham.jw_transfer()  # fermion -> spin

mat2 = ham.to_matrix(basis, pauli=False)
print(np.linalg.norm(mat1 - mat2))

0.0


In [2]:
# automata - fermion

# 验证正确性：
import quante as qt
import numpy as np

op = qt.generate.operas.spin

L = 5
basis = qt.generate.basis.spin_basis(L=L)
ham = op.heisenberg_operator(L=L).expandxy(pauli=True)
mat1 = ham.to_matrix(basis, pauli=True)
ham = ham.jw_transfer(pauli=True)  # spin -> fermion
ham = ham.jw_transfer()  # fermion -> spin
mpo = ham.to_mpo()
mat2 = mpo.to_matrix().numpy()
print(np.linalg.norm(mat1 - mat2))
print(mpo)

0.0
MPO;  torch.float64;  norm: 1.960e+01;  maxbonddim: 5;  device: cpu;
physdim:    2|    2|    2|    2|    2| 
         ----O-----O-----O-----O-----O----
physdim:    2|    2|    2|    2|    2| 
bonddim:  1     4     5     5     5     1
site:        0     1     2     3     4  


## Spinfull Femion

In [3]:
op = qt.generate.operas.spinful_fermion
builder = op.builder()
builder += '+-|', [0, 1], 1.
builder += '|+-', [1, 0], 1.
builder += '+|-', [1, 0], 1.

In [4]:
import quante as qt
op = qt.generate.operas.spinful_fermion

ham = op.Fermi_Hubbard_operator(L=5)
ham.show_string_form()

SpinfulFermionOper (SpinUp | SpinDown) at 0x2aeae8e36e0, 
|   +     -   |   coef. |   -     +   |   coef. | | +     -       coef. | | -     +       coef. |
|-----------------------|-----------------------|-----------------------|-----------------------|
|   0     1      -1.000 |   0     1       1.000 |   0     1      -1.000 |   0     1       1.000 |
|   1     2      -1.000 |   1     2       1.000 |   1     2      -1.000 |   1     2       1.000 |
|   2     3      -1.000 |   2     3       1.000 |   2     3      -1.000 |   2     3       1.000 |
|   3     4      -1.000 |   3     4       1.000 |   3     4      -1.000 |   3     4       1.000 |
|   n   | n       coef. |
|-----------------------|
|   0     0       5.000 |
|   1     1       5.000 |
|   2     2       5.000 |
|   3     3       5.000 |
|   4     4       5.000 |



C:\Users\hzhu\AppData\Local\Temp\ipykernel_32488\1321301931.py:5: DeprecationWarning: show_string_form is deprecated: show_string_form is deprecated, use show instead
  ham.show_string_form()


In [5]:
import quante as qt
op = qt.generate.operas.spinful_fermion

L = 5
builder = op.builder()
for i in range(L-1):
    builder += "+-|", [i, i+1], -1.0
    builder += "-+|", [i, i+1], 1.0
    builder += "|+-", [i, i+1], -1.0
    builder += "|-+", [i, i+1], 1.0
for i in range(L):
    builder += "n|n", [i, i], 5.0
ham = builder.build()
ham


In [6]:
import quante as qt
import numpy as np
op = qt.generate.operas.spinful_fermion

L = 5
builder = op.builder()
for i in range(L-1):
    builder += "+-|", [i, i+1], -1.0
    builder += "-+|", [i, i+1], 1.0
    builder += "|+-", [i, i+1], -1.0
    builder += "|-+", [i, i+1], 1.0
for i in range(L):
    builder += "n|n", [i, i], 5.0
ham = builder.build()

import quante.bridge.quspin_utils as qs
basis = qs.spinful_fermion_basis(L=L)
mat = qs.hamiltonian(ham, basis=basis, dtype=np.float64)
mat

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4877 stored elements and shape (1024, 1024)>

In [7]:
# 展成 spinless fermion
import quante as qt
op = qt.generate.operas.spinful_fermion

L = 5
builder = op.builder()
for i in range(L-1):
    builder += "+-|", [i, i+1], -1.0
    builder += "-+|", [i, i+1], 1.0
    builder += "|+-", [i, i+1], -1.0
    builder += "|-+", [i, i+1], 1.0
for i in range(L):
    builder += "n|n", [i, i], 5.0
ham = builder.build()
ham = ham.to_spinless(mode='extend')
ham

In [8]:
# 展成 spinless fermion
import quante as qt
op = qt.generate.operas.spinful_fermion

L = 5
builder = op.builder()
for i in range(L-1):
    builder += "+-|", [i, i+1], -1.0
    builder += "-+|", [i, i+1], 1.0
    builder += "|+-", [i, i+1], -1.0
    builder += "|-+", [i, i+1], 1.0
for i in range(L):
    builder += "n|n", [i, i], 5.0
ham = builder.build()
ham = ham.to_spinless(mode='extend')

import quante.bridge.quspin_utils as qs
basis = qs.fermion_basis(L=2*L)
mat = qs.hamiltonian(ham, basis=basis, dtype=np.float64, sparse=False)
mat, np.linalg.eigvalsh(mat)[0]

(array([[25.,  0.,  0., ...,  0.,  0.,  0.],
        [ 0., 20., -1., ...,  0.,  0.,  0.],
        [ 0., -1., 20., ...,  0.,  0.,  0.],
        ...,
        [ 0.,  0.,  0., ...,  0., -1.,  0.],
        [ 0.,  0.,  0., ..., -1.,  0.,  0.],
        [ 0.,  0.,  0., ...,  0.,  0.,  0.]]),
 np.float64(-3.3826179609967295))

In [9]:
# 展成 spinless fermion
import quante as qt
op = qt.generate.operas.spinful_fermion

L = 5
builder = op.builder()
for i in range(L-1):
    builder += "+-|", [i, i+1], -1.0
    builder += "-+|", [i, i+1], 1.0
    builder += "|+-", [i, i+1], -1.0
    builder += "|-+", [i, i+1], 1.0
for i in range(L):
    builder += "n|n", [i, i], 5.0
ham = builder.build()
ham = ham.to_spinless(mode='near')

import quante.bridge.quspin_utils as qs
basis = qs.fermion_basis(L=2*L)
mat = qs.hamiltonian(ham, basis=basis, dtype=np.float64, sparse=False)
mat, np.linalg.eigvalsh(mat)[0]

(array([[25.,  0.,  0., ...,  0.,  0.,  0.],
        [ 0., 20.,  0., ...,  0.,  0.,  0.],
        [ 0.,  0., 20., ...,  0.,  0.,  0.],
        ...,
        [ 0.,  0.,  0., ...,  0.,  0.,  0.],
        [ 0.,  0.,  0., ...,  0.,  0.,  0.],
        [ 0.,  0.,  0., ...,  0.,  0.,  0.]]),
 np.float64(-3.382617960996767))